# Seed-Guided Dual-Encoder Prototype Matching

This notebook implements the method described in `SEED_DESCRIPTION.pdf`.

Pipeline:
1. Load trusted ImgFlip seed folders with `DataLoader`.
2. Load LLM-annotated targets from `data/merged_parsed_results.jsonl`.
3. Embed seed and target images with SigLIP2 and DINOv2.
4. Build one prototype per template for each encoder.
5. Run phase-1 assignment using fused cosine similarity, `tau`, and `delta`.
6. Update prototypes with pseudo-label weight `beta`.
7. Run a second scoring pass and save final predictions.


In [88]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, Dinov2Model, SiglipVisionModel


In [89]:
def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def first_existing(paths: Iterable[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def find_project_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve().parent.parent]
    for candidate in candidates:
        if (candidate / 'python_scripts').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root. Start Jupyter from the repo or set PROJECT_ROOT manually.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [90]:
from python_scripts.data_loader import DataLoader

@dataclass
class SeedGuidedConfig:
    seed_root: Path
    image_root: Path
    target_table_path: Path
    output_path: Path
    key_col: str = "key"
    reference_label_col: str = "original_template"   # or "template"
    no_template_label: str = "NO_TEMPLATE"
    siglip_model_id: str = "google/siglip2-base-patch16-224"
    dino_model_id: str = "facebook/dinov2-base"
    batch_size: int = 8
    alpha: float = 0.60
    tau: float = 0.85
    delta: float = 0.1
    beta: float = 0.15
    max_templates: int | None = None
    max_targets: int | None = None


DEVICE = pick_device()
MODEL_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

SEED_ROOT = first_existing([
    PROJECT_ROOT / "IMGFlip2024_haslabel",
    Path("/Volumes/huysuy05/ssd_data/meme_or_not/IMGFlip2024_haslabel"),
])
IMAGE_ROOT = first_existing([
    PROJECT_ROOT / "Reddit2024_nolabel" / "images",
])

if SEED_ROOT is None:
    raise FileNotFoundError("ImgFlip seed folder not found. Update SEED_ROOT in this cell.")
if IMAGE_ROOT is None:
    raise FileNotFoundError("Target image folder not found. Update IMAGE_ROOT in this cell.")

cfg = SeedGuidedConfig(
    seed_root=SEED_ROOT,
    image_root=IMAGE_ROOT,
    target_table_path=PROJECT_ROOT / "SEED" / "meme_data_reassigned_templates.csv",
    output_path=PROJECT_ROOT / "SEED" / "seed_guided_predictions.csv",
    reference_label_col='original_template',
)


In [91]:
def l2_normalize(array: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(array, axis=-1, keepdims=True)
    return array / np.clip(norms, 1e-12, None)


def iter_batches(items: list[Any], batch_size: int):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


from PIL import Image, ImageOps, UnidentifiedImageError

def load_rgb_image(path: str | Path) -> Image.Image | None:
    try:
        with Image.open(path) as img:
            return ImageOps.exif_transpose(img).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError):
        return None



def resolve_target_image_path(key: str, image_root: Path) -> str | None:
    for ext in ['.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif']:
        candidate = image_root / f'{key}{ext}'
        if candidate.exists():
            return str(candidate)
    return None


def extract_nested(data: dict | None, *keys: str) -> Any:
    cur: Any = data
    for key in keys:
        if not isinstance(cur, dict):
            return None
        cur = cur.get(key)
    return cur


def build_seed_index(loader: DataLoader, max_templates: int | None = None) -> pd.DataFrame:
    rows = []
    template_names = loader.get_template_names()
    if max_templates is not None:
        template_names = template_names[:max_templates]

    for template_name in template_names:
        template_dir = loader.root_dir / template_name
        image_paths = loader._images_in_dir(template_dir)
        for image_path in image_paths:
            rows.append({
                'template': template_name,
                'image_path': str(image_path),
            })

    if not rows:
        raise ValueError(f'No seed images found under {loader.root_dir}')

    return pd.DataFrame(rows)


def load_target_table(cfg: SeedGuidedConfig) -> pd.DataFrame:
    path = cfg.target_table_path

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        df[cfg.key_col] = df[cfg.key_col].astype(str).str.strip()
        df["image_path"] = df[cfg.key_col].map(lambda key: resolve_target_image_path(key, cfg.image_root))

        keep_cols = [cfg.key_col, cfg.reference_label_col, "image_path"]
        optional_cols = [
            "template",
            "original_template",
            "matched_template",
            "match_confidence",
            "embedding_score",
            "lexical_score",
            "global_context_description",
            "local_context_user_texts",
            "local_context_text_meaning",
            "local_context_instance_specific_image_description",
        ]
        keep_cols.extend([col for col in optional_cols if col in df.columns])

        # remove duplicates while preserving order
        keep_cols = list(dict.fromkeys(keep_cols))

        df = df[keep_cols]
        df = df[df["image_path"].notna()].copy()

        df[cfg.reference_label_col] = (
            df[cfg.reference_label_col]
            .fillna(cfg.no_template_label)
            .astype(str)
            .str.strip()
        )

        if cfg.max_targets is not None:
            df = df.head(cfg.max_targets).copy()

        return df.reset_index(drop=True)

    # original JSONL path
    df = pd.read_json(path, lines=True)
    df[cfg.key_col] = df[cfg.key_col].astype(str).str.strip()

    df[cfg.reference_label_col] = df["data"].map(lambda value: extract_nested(value, "template"))
    df["llm_global_context_description"] = df["data"].map(lambda value: extract_nested(value, "global_context_description"))
    df["llm_user_texts"] = df["data"].map(lambda value: extract_nested(value, "local_context", "user_texts"))
    df["llm_text_meaning"] = df["data"].map(lambda value: extract_nested(value, "local_context", "text_meaning"))
    df["llm_instance_specific_image_description"] = df["data"].map(lambda value: extract_nested(value, "local_context", "instance_specific_image_description"))
    df["image_path"] = df[cfg.key_col].map(lambda key: resolve_target_image_path(key, cfg.image_root))

    keep_cols = [
        cfg.key_col,
        cfg.reference_label_col,
        "llm_global_context_description",
        "llm_user_texts",
        "llm_text_meaning",
        "llm_instance_specific_image_description",
        "image_path",
    ]
    df = df[keep_cols]
    df = df[df["image_path"].notna()].copy()
    df[cfg.reference_label_col] = df[cfg.reference_label_col].fillna(cfg.no_template_label).astype(str).str.strip()

    if cfg.max_targets is not None:
        df = df.head(cfg.max_targets).copy()

    return df.reset_index(drop=True)


In [92]:
class HFImageEmbedder:
    def __init__(self, model_id: str, processor_cls, model_cls, batch_size: int):
        self.model_id = model_id
        self.batch_size = batch_size
        self.processor = processor_cls.from_pretrained(model_id)
        self.model = model_cls.from_pretrained(model_id, torch_dtype=MODEL_DTYPE).to(DEVICE)
        self.model.eval()

    @torch.inference_mode()
    def encode_paths(self, paths: list[str]) -> tuple[np.ndarray, list[str]]:
        if not paths:
            raise ValueError(f'No paths passed to {self.model_id}')

        vectors = []
        kept_paths = []

        for batch_paths in tqdm(iter_batches(paths, self.batch_size), desc=f'embed {self.model_id}'):
            loaded = []
            valid_paths = []

            for path in batch_paths:
                img = load_rgb_image(path)
                if img is not None:
                    loaded.append(img)
                    valid_paths.append(path)

            if not loaded:
                continue

            inputs = self.processor(images=loaded, return_tensors='pt')
            inputs = {name: tensor.to(DEVICE) for name, tensor in inputs.items()}
            outputs = self.model(**inputs)

            pooled = getattr(outputs, 'pooler_output', None)
            if pooled is None:
                pooled = outputs.last_hidden_state[:, 0]

            pooled = F.normalize(pooled.float(), dim=-1)
            vectors.append(pooled.cpu())
            kept_paths.extend(valid_paths)

        if not vectors:
            raise ValueError(f'All images failed to load for {self.model_id}')

        return torch.cat(vectors, dim=0).numpy(), kept_paths



In [93]:
def build_prototypes(template_names: list[str], row_templates: np.ndarray, embeddings: np.ndarray) -> np.ndarray:
    prototypes = []
    for template_name in template_names:
        mask = row_templates == template_name
        if not mask.any():
            raise ValueError(f'No seed rows found for template={template_name}')
        centroid = embeddings[mask].mean(axis=0, keepdims=True)
        prototypes.append(l2_normalize(centroid)[0])
    return np.vstack(prototypes)


def fused_scores(target_siglip: np.ndarray, target_dino: np.ndarray, proto_siglip: np.ndarray, proto_dino: np.ndarray, alpha: float) -> np.ndarray:
    siglip_scores = target_siglip @ proto_siglip.T
    dino_scores = target_dino @ proto_dino.T
    return alpha * siglip_scores + (1.0 - alpha) * dino_scores


def assign_from_scores(scores: np.ndarray, tau: float, delta: float) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    best_idx = scores.argmax(axis=1)
    row_ids = np.arange(scores.shape[0])
    best_scores = scores[row_ids, best_idx]

    if scores.shape[1] == 1:
        second_scores = np.full(scores.shape[0], -np.inf, dtype=scores.dtype)
    else:
        masked = scores.copy()
        masked[row_ids, best_idx] = -np.inf
        second_scores = masked.max(axis=1)

    margins = best_scores - second_scores
    accepted = (best_scores >= tau) & (margins >= delta)
    assigned_idx = np.where(accepted, best_idx, -1)
    return assigned_idx, best_idx, best_scores, second_scores, margins


def update_prototypes(template_names: list[str], seed_templates: np.ndarray, seed_embeddings: np.ndarray, target_embeddings: np.ndarray, assigned_idx: np.ndarray, beta: float) -> np.ndarray:
    updated = []
    for template_idx, template_name in enumerate(template_names):
        seed_mask = seed_templates == template_name
        weighted_sum = seed_embeddings[seed_mask].sum(axis=0)
        total_weight = float(seed_mask.sum())

        pseudo_mask = assigned_idx == template_idx
        if pseudo_mask.any():
            weighted_sum = weighted_sum + beta * target_embeddings[pseudo_mask].sum(axis=0)
            total_weight += beta * float(pseudo_mask.sum())

        prototype = (weighted_sum / total_weight)[None, :]
        updated.append(l2_normalize(prototype)[0])

    return np.vstack(updated)


def template_lookup_with_reference_fallback(
    template_names: list[str] | np.ndarray,
    assigned_idx: np.ndarray,
    reference_labels: pd.Series,
    no_template_label: str,
) -> list[str]:
    clean_reference = (
        reference_labels
        .fillna(no_template_label)
        .astype(str)
        .str.strip()
        .replace('', no_template_label)
    )
    output = clean_reference.tolist()
    for row_idx, tpl_idx in enumerate(assigned_idx):
        if tpl_idx >= 0:
            output[row_idx] = str(template_names[tpl_idx])
    return output



In [94]:
def run_seed_guided_matching(cfg: SeedGuidedConfig) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    loader = DataLoader(str(cfg.seed_root))
    seed_df = build_seed_index(loader, cfg.max_templates)
    target_df = load_target_table(cfg)

    from PIL import Image

    def filter_valid_image_rows(df, path_col="image_path"):
        keep = []
        for path in df[path_col]:
            try:
                with Image.open(path) as img:
                    img.verify()
                keep.append(True)
            except Exception:
                keep.append(False)
        return df[keep].reset_index(drop=True)


    print('seed images:', len(seed_df))
    print('target images:', len(target_df))
    print('templates used:', seed_df['template'].nunique())

    siglip = HFImageEmbedder(cfg.siglip_model_id, AutoImageProcessor, SiglipVisionModel, cfg.batch_size)
    dino = HFImageEmbedder(cfg.dino_model_id, AutoImageProcessor, Dinov2Model, cfg.batch_size)




    seed_paths = seed_df['image_path'].tolist()
    target_paths = target_df['image_path'].tolist()
    seed_templates = seed_df['template'].to_numpy()
    template_names = np.array(sorted(seed_df['template'].unique().tolist()))


    cache_dir = PROJECT_ROOT / "SEED" / "embedding_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)

    seed_siglip_path = cache_dir / "seed_siglip.npy"
    seed_dino_path = cache_dir / "seed_dino.npy"
    target_siglip_path = cache_dir / "target_siglip.npy"
    target_dino_path = cache_dir / "target_dino.npy"
    seed_df_path = cache_dir / "seed_df.parquet"
    target_df_path = cache_dir / "target_df.parquet"
    template_names_path = cache_dir / "template_names.npy"

    template_names = np.array(sorted(seed_df["template"].unique().tolist()))

    if all(path.exists() for path in [
        seed_siglip_path,
        seed_dino_path,
        target_siglip_path,
        target_dino_path,
        seed_df_path,
        target_df_path,
        template_names_path,
    ]):
        print("Loading cached embeddings...")
        seed_df = pd.read_parquet(seed_df_path)
        target_df = pd.read_parquet(target_df_path)
        seed_siglip = np.load(seed_siglip_path)
        seed_dino = np.load(seed_dino_path)
        target_siglip = np.load(target_siglip_path)
        target_dino = np.load(target_dino_path)
        template_names = np.load(template_names_path, allow_pickle=True)
        template_names = np.array(template_names)
        seed_templates = seed_df["template"].to_numpy()
    else:
        print("Encoding embeddings from scratch...")
        seed_siglip, seed_paths_kept_siglip = siglip.encode_paths(seed_paths)
        seed_dino, seed_paths_kept_dino = dino.encode_paths(seed_paths)
        target_siglip, target_paths_kept_siglip = siglip.encode_paths(target_paths)
        target_dino, target_paths_kept_dino = dino.encode_paths(target_paths)

        if seed_paths_kept_siglip != seed_paths_kept_dino:
            raise ValueError("Seed kept-path mismatch between SigLIP and DINO encoders.")
        if target_paths_kept_siglip != target_paths_kept_dino:
            raise ValueError("Target kept-path mismatch between SigLIP and DINO encoders.")

        if len(seed_paths_kept_siglip) != len(seed_df):
            seed_df = seed_df.set_index("image_path").loc[seed_paths_kept_siglip].reset_index()
        if len(target_paths_kept_siglip) != len(target_df):
            target_df = target_df.set_index("image_path").loc[target_paths_kept_siglip].reset_index()

        seed_templates = seed_df["template"].to_numpy()
        template_names = np.array(sorted(seed_df["template"].unique().tolist()))

        seed_df.to_parquet(seed_df_path, index=False)
        target_df.to_parquet(target_df_path, index=False)

        np.save(seed_siglip_path, seed_siglip)
        np.save(seed_dino_path, seed_dino)
        np.save(target_siglip_path, target_siglip)
        np.save(target_dino_path, target_dino)
        np.save(template_names_path, np.array(template_names, dtype=object))


    proto_siglip = build_prototypes(template_names.tolist(), seed_templates, seed_siglip)
    proto_dino = build_prototypes(template_names.tolist(), seed_templates, seed_dino)


    phase1_scores = fused_scores(target_siglip, target_dino, proto_siglip, proto_dino, cfg.alpha)
    phase1_assigned, phase1_best_idx, phase1_best, phase1_second, phase1_margin = assign_from_scores(phase1_scores, cfg.tau, cfg.delta)

    proto_siglip_new = update_prototypes(template_names, seed_templates, seed_siglip, target_siglip, phase1_assigned, cfg.beta)
    proto_dino_new = update_prototypes(template_names, seed_templates, seed_dino, target_dino, phase1_assigned, cfg.beta)

    phase2_scores = fused_scores(target_siglip, target_dino, proto_siglip_new, proto_dino_new, cfg.alpha)
    phase2_assigned, phase2_best_idx, phase2_best, phase2_second, phase2_margin = assign_from_scores(phase2_scores, cfg.tau, cfg.delta)

    results = target_df.copy()
    results['phase1_best_template'] = template_names[phase1_best_idx]

    results['phase1_matched_imgflip'] = phase1_assigned >= 0
    results['phase1_template'] = template_lookup_with_reference_fallback(
        template_names,
        phase1_assigned,
        results[cfg.reference_label_col],
        cfg.no_template_label,
    )
    results['phase1_best_score'] = phase1_best
    results['phase1_second_score'] = phase1_second
    results['phase1_margin'] = phase1_margin

    results['final_best_template'] = template_names[phase2_best_idx]
    results['final_matched_imgflip'] = phase2_assigned >= 0
    results['final_template'] = template_lookup_with_reference_fallback(
        template_names,
        phase2_assigned,
        results[cfg.reference_label_col],
        cfg.no_template_label,
    )
    results['final_best_score'] = phase2_best
    results['final_second_score'] = phase2_second
    results['final_margin'] = phase2_margin

    return results, seed_df, template_names


In [95]:
# For a smoke test, consider setting smaller limits first:
# cfg.max_templates = 100
# cfg.max_targets = 250

results_df, seed_df, template_names = run_seed_guided_matching(cfg)
results_df.to_csv(cfg.output_path, index=False)

print('saved predictions to:', cfg.output_path)
print('phase1 matched imgflip:', int(results_df['phase1_matched_imgflip'].sum()))
print('final matched imgflip:', int(results_df['final_matched_imgflip'].sum()))

results_df[[cfg.key_col, cfg.reference_label_col, 'phase1_template', 'phase1_best_score', 'phase1_margin', 'final_template', 'final_best_score', 'final_margin']].head()


/var/folders/q9/sf8tjzc130s4gxh60ngggj4w0000gn/T/ipykernel_17323/292693155.py:64: DtypeWarning: Columns (10,12,13,14,15,16,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


seed images: 142240
target images: 169436
templates used: 1152


Loading weights: 100%|██████████| 208/208 [00:00<00:00, 1635.73it/s, Materializing param=vision_model.post_layernorm.weight]                      
SiglipVisionModel LOAD REPORT from: google/siglip2-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UN

Loading cached embeddings...
saved predictions to: /Users/huy.suy05./Documents/Projects/To_Meme_or_To_Not_Meme/SEED/seed_guided_predictions.csv
phase1 matched imgflip: 28435
final matched imgflip: 29092


,key,original_template,phase1_template,phase1_best_score,phase1_margin,final_template,final_best_score,final_margin
0,meme_submissions_1343519,NO_TEMPLATE,Loading-cat,0.857572,0.198754,Loading-cat,0.860763,0.201889
1,meme_submissions_134352,I fear no man. But that thing... it scares me.,I-fear-no-man-But-that-thingit-scares-me,0.887566,0.138422,I-fear-no-man-But-that-thingit-scares-me,0.886690,0.137546
2,meme_submissions_1343524,NO_TEMPLATE,NO_TEMPLATE,0.597307,0.003666,NO_TEMPLATE,0.597302,0.003661
3,meme_submissions_1343526,"Homer Simpson ""Something so stupid""","Homer Simpson ""Something so stupid""",0.668337,0.011764,"Homer Simpson ""Something so stupid""",0.668337,0.011539
4,meme_submissions_134353,NO_TEMPLATE,NO_TEMPLATE,0.599890,0.017125,NO_TEMPLATE,0.602741,0.018947


In [81]:
if cfg.reference_label_col in results_df.columns:
    eval_df = results_df.copy()
    eval_df[cfg.reference_label_col] = eval_df[cfg.reference_label_col].fillna(cfg.no_template_label).astype(str).str.strip()

    in_scope_labels = set(template_names) | {cfg.no_template_label}
    in_scope_mask = eval_df[cfg.reference_label_col].isin(in_scope_labels)
    eval_scope = eval_df[in_scope_mask].copy()

    print('rows with in-scope LLM labels:', len(eval_scope))
    print('phase1 agreement vs llm_template:', (eval_scope['phase1_template'] == eval_scope[cfg.reference_label_col]).mean())
    print('final agreement vs llm_template:', (eval_scope['final_template'] == eval_scope[cfg.reference_label_col]).mean())

    eval_scope[[cfg.reference_label_col, 'phase1_template', 'final_template']].head()
else:
    print(f'{cfg.reference_label_col} not found; skipping agreement check.')


rows with in-scope LLM labels: 91028
phase1 agreement vs llm_template: 0.8676670914443907
final agreement vs llm_template: 0.8442347409588259


In [82]:
results_df['final_template'].value_counts().head(20)


final_template
NO_TEMPLATE                             117664
drake-yes-no-reverse                      1777
Google-Translate                          1449
Change-My-Mind                             764
Giga-Chad                                  610
Grus-Plan                                  520
thanos-impossible                          510
AJ-Styles--Undertaker                      504
Monkey-looking-away                        500
Lisa-Simpsons-Presentation                 480
They-are-the-same-picture                  473
Angry-npc-wojak                            447
boys-vs-girls                              431
Strong-dog-vs-weak-dog                     430
train-crashes-bus                          427
Vince-McMahon-Reaction-wGlowing-Eyes       427
stonks                                     407
Gus-Fring-we-are-not-the-same              400
Anakin-and-Padme                           394
Afraid-To-Ask-Andy                         391
Name: count, dtype: int64